In [ ]:
import collections
import os
import re

import numpy as np
import cv2
import yaml
# import PIL
# import PIL.Image
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, ConfusionMatrixDisplay
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
import tensorflow.keras.backend as K
from tensorflow.keras.optimizers import Adam


In [ ]:
# 精度がばらけるので再現性確認のため、cpuだけで学習する
# tf.config.set_visible_devices([], "GPU")

In [ ]:
# yamlの設定ファイルの読み込み

with open("../config/config.yaml", "rb") as f:
    config = yaml.safe_load(f)
    # print(config)
    # print("launch json modified")
    # print("mapping_namelist", config["class_mapping"]["name_list"])


In [ ]:
# dir_data = config["path_extracted_data"]

dir_path_train = config["path_split_extracted_data"]["train"]
dir_path_val = config["path_split_extracted_data"]["val"]
dir_path_test = config["path_split_extracted_data"]["common_test"]

# 画像サイズは、efficientnet b0の想定
img_height = 224
img_width = 224

seed = 42
batch_size = 16

split_rate = 0.2

epoch = 15


1.画像データの読み込み

In [ ]:
# 学習データの読み込み
dataset_train = tf.keras.utils.image_dataset_from_directory(
    dir_path_train,
    # validation_split = split_rate,
    # subset = "training",    # 学習用    
    seed = seed, 
    image_size = (img_height, img_width),
    batch_size=batch_size,
    shuffle=False,
)

In [ ]:
# 検証データの読み込み
dataset_val = tf.keras.utils.image_dataset_from_directory(
    dir_path_val,
    # validation_split = split_rate,
    # subset = "val",    # 検証用    
    seed = seed, 
    image_size = (img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# 共通テストデータの読み込み
dataset_test = tf.keras.utils.image_dataset_from_directory(
    dir_path_test,
    # validation_split = split_rate,
    # subset = "test",    # テスト用    
    seed = seed, 
    image_size = (img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

2.モデルの構築,学習,評価

In [ ]:
# 乱数管理とメモリ解放のために１セルにまとめる

tf.keras.utils.set_random_seed(seed)

# TensorFlowの「決定論的動作」。
tf.config.experimental.enable_op_determinism()

# top層を除いた、学習済みモデルの読み込み
base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(img_height, img_width, 3))

print("efficient net loaded")

# トップ層のみ学習したいため、層を凍結
base_model.trainable = False

# モデルの構築
inputs = layers.Input(shape=(img_height, img_width, 3))
x = base_model(inputs, training=False) #base_model.trainable=Falseしてるけど、batch nomarization層を固定するためにここでもfalse
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(len(dataset_train.class_names), activation="softmax")(x) #多分類なのでsoftmax

model_baseline = Model(inputs, outputs)

model_baseline.compile(
    optimizer="adam",
    # optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# print(np.sum([np.sum(w) for w in model_baseline.get_weights()]))

print("train stats")

model_baseline.fit(
    x=dataset_train, 
    epochs=epoch, 
    # epochs=1, #動作確認用
    validation_data=dataset_val,
    shuffle=False
    )

print("\npred starts")

# tf.keras.utils.image_dataset_from_directoryでデータセットを作ると、ディレクトリ名がそのまま正解ラベルになる。
# けれど、predictでは、正解ラベルを無視して予測してくれる
y_probs = model_baseline.predict(dataset_test)

# 各カテゴリの確率のうち、最も高い確率のカテゴリを予測されたカテゴリとする
y_pred = np.argmax(y_probs, axis=-1)

# datasetのバッチごとに、xが画像の要素、yがラベル。yを正解ラベルとして取り出す。
y_true = np.concatenate([y for x, y in dataset_test], axis=0)

acc = accuracy_score(y_true, y_pred)
print(f"acc: {acc}")

K.clear_session()

In [ ]:
# 混合行列の確認
confusion_matrix(y_true, y_pred)

In [ ]:
# 全クラスの平均をとる
f1_score(y_true, y_pred, average="macro")

In [ ]:
classification_report(y_true, y_pred, target_names=dataset_test.class_names)

In [ ]:
# 混合行列のグラフ化
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Confusion Matrix")
plt.show()